# Video Indexing at Speed - Kaggle Execution Notebook

### Instructions:
1. **Settings -> Accelerator**: Set to **GPU T4 x 2** (Dual T4 for 2-GPU sharding) or **GPU P100**.
2. **Settings -> Internet**: Toggle **ON**.
3. **HuggingFace Access**: Accept dataset terms at [huggingface.co/datasets/HuggingFaceFV/finevideo](https://huggingface.co/datasets/HuggingFaceFV/finevideo) and create a Read Token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [ ]:
import glob, os, shutil, subprocess

# 1. Ensure working directory is /kaggle/working
%cd /kaggle/working

print("=== Setting up code repository in /kaggle/working ===")

def is_ready():
    return os.path.exists("/kaggle/working/run.py") and \
           os.path.exists("/kaggle/working/requirements.txt") and \
           os.path.isdir("/kaggle/working/vidindex")

if not is_ready():
    # Strategy A: Check if a zip file was uploaded in /kaggle/input
    zip_files = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if zip_files:
        print(f"Found zip archive: {zip_files[0]}, extracting...")
        subprocess.run(["unzip", "-q", "-o", zip_files[0], "-d", "/kaggle/working"], check=True)
        # If files were extracted into a nested directory, flatten them to /kaggle/working
        nested_run = glob.glob("/kaggle/working/**/run.py", recursive=True)
        if nested_run and os.path.dirname(nested_run[0]) != "/kaggle/working":
            sub = os.path.dirname(nested_run[0])
            print(f"Flattening nested folder: {sub} -> /kaggle/working")
            for item in os.listdir(sub):
                src = os.path.join(sub, item)
                dst = os.path.join("/kaggle/working", item)
                if os.path.exists(dst):
                    if os.path.isdir(dst): shutil.rmtree(dst)
                    else: os.remove(dst)
                shutil.move(src, dst)

    # Strategy B: Check if files were extracted into /kaggle/input
    if not is_ready():
        run_paths = glob.glob("/kaggle/input/**/run.py", recursive=True)
        if run_paths:
            src_dir = os.path.dirname(run_paths[0])
            print(f"Found source files in dataset at: {src_dir}")
            for item in os.listdir(src_dir):
                s = os.path.join(src_dir, item)
                d = os.path.join("/kaggle/working", item)
                if os.path.isdir(s):
                    if os.path.exists(d): shutil.rmtree(d)
                    shutil.copytree(s, d)
                else:
                    shutil.copy2(s, d)

    # Strategy C: If no dataset is attached at all, clone directly from GitHub
    if not is_ready():
        print("No local dataset found in /kaggle/input. Cloning repository directly from GitHub...")
        subprocess.run(["git", "clone", "https://github.com/ChishtiShaheer/Video-Indexer.git", "/tmp/repo"], check=True)
        for item in os.listdir("/tmp/repo"):
            if item == ".git": continue
            s = os.path.join("/tmp/repo", item)
            d = os.path.join("/kaggle/working", item)
            if os.path.isdir(s):
                if os.path.exists(d): shutil.rmtree(d)
                shutil.copytree(s, d)
            else:
                shutil.copy2(s, d)

# Verification
assert os.path.exists("/kaggle/working/run.py"), "❌ run.py missing from /kaggle/working!"
assert os.path.exists("/kaggle/working/requirements.txt"), "❌ requirements.txt missing from /kaggle/working!"
assert os.path.isdir("/kaggle/working/vidindex"), "❌ vidindex/ directory missing from /kaggle/working!"

print("\n✅ Codebase setup complete in /kaggle/working:")
for f in sorted(os.listdir("/kaggle/working")):
    if not f.startswith("."):
        print(f"  - {f}")

print("\n--- Verifying detector.py compatibility ---")
!grep -n "predict_kwargs" /kaggle/working/vidindex/detector.py && echo "✅ Detector compatibility verified!" || echo "⚠️ Detector file check"


In [ ]:
# Install dependencies from requirements.txt
import os, glob

req_file = "/kaggle/working/requirements.txt"
if not os.path.exists(req_file):
    matches = glob.glob("/kaggle/**/requirements.txt", recursive=True)
    if matches:
        req_file = matches[0]
        print(f"Found requirements.txt at: {req_file}")

print(f"Installing dependencies using {req_file}...")
!pip install -q --upgrade pip
!pip install -q -r "{req_file}"
print("\n✅ All packages installed successfully.")


In [ ]:
# Step 2: Run unit tests to verify the pipeline code is intact
!python /kaggle/working/run.py --help > /dev/null && echo '✅ run.py OK'
!python -m unittest discover -s /kaggle/working/tests -v


In [ ]:
# Step 3: Download YOLO weights and warm up model
!python /kaggle/working/run.py --prepare --batch-size 64


In [ ]:
# Step 4: HuggingFace login
# Replace [REMOVED] with your HF read token from huggingface.co/settings/tokens
from huggingface_hub import login
login(token="[REMOVED]")


In [ ]:
%%bash
echo '=== /kaggle/working space (20GB limited) ==='
df -h /kaggle/working

echo ''
echo '=== /tmp space (NO quota, separate partition) ==='
df -h /tmp

echo ''
echo 'Downloading all 50 parquet files into /tmp/finevideo/data/ ...'
mkdir -p /tmp/finevideo/data

hf download HuggingFaceFV/finevideo \
  --repo-type dataset \
  --include 'data/train-000[0-4]*.parquet' \
  --local-dir /tmp/finevideo

echo ''
echo 'Total files downloaded:'
ls /tmp/finevideo/data/*.parquet | wc -l

echo 'Disk check after download:'
df -h /tmp


In [ ]:
%%bash
pkill -9 -f "run.py" || true
rm -f /kaggle/working/shard0.log /kaggle/working/shard1.log
rm -f /kaggle/working/results_shard0.jsonl /kaggle/working/results_shard1.jsonl

python /kaggle/working/run.py \
  --input /tmp/finevideo/data \
  --output /kaggle/working/results_shard0.jsonl \
  --shard-index 0 --num-shards 2 \
  --device cuda:0 --workers 0 \
  --fps 5 --batch-size 64 \
  > /kaggle/working/shard0.log 2>&1 &

python /kaggle/working/run.py \
  --input /tmp/finevideo/data \
  --output /kaggle/working/results_shard1.jsonl \
  --shard-index 1 --num-shards 2 \
  --device cuda:1 --workers 0 \
  --fps 5 --batch-size 64 \
  > /kaggle/working/shard1.log 2>&1 &

echo "Launched. PIDs: $(pgrep -f run.py | tr '\n' ' ')"
wait
echo "=== Both done! ==="

In [ ]:
%%bash
echo "=== 1. Check ALL .jsonl files in /kaggle/working ==="
ls -lh /kaggle/working/*.jsonl 2>/dev/null || echo "No .jsonl files found"
wc -l /kaggle/working/*.jsonl 2>/dev/null

echo ""
echo "=== 2. Check if processes are running & CPU/GPU usage ==="
ps aux | grep run.py | grep -v grep || echo "NO run.py process running!"

echo ""
echo "=== 3. FULL shard0.log (last 40 lines) ==="
tail -n 40 /kaggle/working/shard0.log 2>/dev/null || echo "No shard0.log"

echo ""
echo "=== 4. FULL shard1.log (last 40 lines) ==="
tail -n 40 /kaggle/working/shard1.log 2>/dev/null || echo "No shard1.log"

In [ ]:
from IPython.display import FileLink, display

print("=== 1. Checking all shard files ===")
!ls -lh /kaggle/working/*.jsonl
!wc -l /kaggle/working/*.jsonl

print("\n=== 2. Merging results_shard0 & results_shard1 ===")
!rm -f /kaggle/working/results.jsonl
!cat /kaggle/working/results_shard0.shard0of2.jsonl /kaggle/working/results_shard1.shard1of2.jsonl > /kaggle/working/results.jsonl
!wc -l /kaggle/working/results.jsonl

print("\n=== 3. Compressing to results.jsonl.gz ===")
!gzip -k -9 -f /kaggle/working/results.jsonl
!ls -lh /kaggle/working/results.jsonl.gz

print("\n=== 4. Download Link ===")
display(FileLink('results.jsonl.gz'))